# Combine 2D correction votes

Normalize every available charge–$p_T$ vote histogram to unit integral, add them with equal weight, and extract one common $(\Delta r, r\Delta\phi)$ correction per detector bin.

In [1]:
import ROOT as root
import math
from pathlib import Path

#root.gROOT.SetBatch(True)
%jsroot on


Welcome to JupyROOT 6.30/06


In [ ]:

input_file = Path('input/pp_test3.root')
output_file = Path('cluster_2d_correction_map.root')

write_output = False
peak_radius_bins = 2
min_hist_integral = 1.0
min_categories = 6
require_both_charges = True

sides = ['side0', 'side1']
charges = ['qplus', 'qminus']
pt_bins = ['pt_0p2_0p4','pt_0p4_0p7','pt_0p7_1p2','pt_1p2_1p8','pt_1p8_5p0']
#pt_bins = ['pt_0p2_0p4','pt_0p4_0p7','pt_0p7_1p2']


In [3]:
def get_hist(root_file, side, charge, pt_bin, module, phi_bin, layer_bin):
    path = (f'{side}/{charge}/{pt_bin}/module_{module:02d}/'
            f'phi_{phi_bin}_layer_{layer_bin}/h_deltaRPhi_vs_deltaR_votes')
    return root_file.Get(path)


def clone_normalized(hist, name):
    if not hist or hist.Integral() < min_hist_integral:
        return None
    out = hist.Clone(name)
    out.SetDirectory(0)
    out.Scale(1.0 / out.Integral())
    return out


def weighted_peak_2d(hist, radius_bins=2):
    if not hist or hist.Integral() <= 0:
        return None

    max_bin = hist.GetMaximumBin()
    nx = hist.GetNbinsX() + 2
    ny = hist.GetNbinsY() + 2
    ix = max_bin % nx
    iy = (max_bin // nx) % ny

    if not (1 <= ix <= hist.GetNbinsX() and 1 <= iy <= hist.GetNbinsY()):
        return None

    bx1, bx2 = max(1, ix-radius_bins), min(hist.GetNbinsX(), ix+radius_bins)
    by1, by2 = max(1, iy-radius_bins), min(hist.GetNbinsY(), iy+radius_bins)

    sx = sy = sw = 0.0
    for bx in range(bx1, bx2+1):
        for by in range(by1, by2+1):
            w = hist.GetBinContent(bx, by)
            if w <= 0:
                continue
            sx += w * hist.GetXaxis().GetBinCenter(bx)
            sy += w * hist.GetYaxis().GetBinCenter(by)
            sw += w

    if sw <= 0:
        return None
    return {'dr': sx/sw, 'drphi': sy/sw, 'maximum': hist.GetMaximum()}


In [4]:
f = root.TFile.Open(str(input_file), 'READ')
if not f or f.IsZombie():
    raise OSError(f'Cannot open {input_file}')

combined_hists = {}
combined_points = {}
category_counts = {}

for side in sides:
    for module in range(36):
        for phi_bin in range(3):
            for layer_bin in range(5):
                combined = None
                used = 0
                used_charges = set()

                for pt_bin in pt_bins:
                    for charge in charges:
                        h = get_hist(f, side, charge, pt_bin, module, phi_bin, layer_bin)
                        hn = clone_normalized(h, f'hn_{side}_{charge}_{pt_bin}_{module}_{phi_bin}_{layer_bin}')
                        if not hn:
                            continue
                        if combined is None:
                            combined = hn.Clone(f'hc_{side}_{module}_{phi_bin}_{layer_bin}')
                            combined.SetDirectory(0)
                        else:
                            combined.Add(hn)
                        used += 1
                        used_charges.add(charge)

                key = (side, module, phi_bin, layer_bin)
                category_counts[key] = used
                valid = (combined is not None and used >= min_categories and
                         (not require_both_charges or len(used_charges) == 2))

                if not valid:
                    combined_hists[key] = None
                    combined_points[key] = None
                    continue

                combined.Scale(1.0 / used)
                combined_hists[key] = combined
                combined_points[key] = weighted_peak_2d(combined, peak_radius_bins)

n_good = sum(v is not None for v in combined_points.values())
print(f'Extracted common peaks in {n_good} / {len(combined_points)} detector bins')


Extracted common peaks in 1056 / 1080 detector bins


## Inspect one detector bin

The following cell draws cumulative 2D maps one below another. The order is: positive charge, opposite charge, then the next $p_T$ bin for both charges, continuing through all bins. Every input histogram is normalized to unit integral before addition.

In [15]:
selected_side = 'side0'
selected_module = 10
selected_phi_bin = 2
selected_layer_bin = 2

draw_peak = True
inspection_objects = []
cumulative = None

for pt_bin in pt_bins:
    for charge in charges:
        h = get_hist(f, selected_side, charge, pt_bin,
                     selected_module, selected_phi_bin, selected_layer_bin)
        hn = clone_normalized(h, f'hinspect_{charge}_{pt_bin}')
        if not hn:
            print(f'Missing: {charge}, {pt_bin}')
            continue

        if cumulative is None:
            cumulative = hn.Clone('h_cumulative')
            cumulative.SetDirectory(0)
        else:
            cumulative.Add(hn)

        shown = cumulative.Clone(f'hshown_{charge}_{pt_bin}')
        shown.SetDirectory(0)
        shown.Scale(1.0 / shown.Integral())
        shown.SetTitle(f'{selected_side}, module {selected_module:02d}, '
                       f'#phi bin {selected_phi_bin}, layer bin {selected_layer_bin};'
                       '#Delta r [cm];r#Delta#phi [cm]')

        c = root.TCanvas(f'c_{charge}_{pt_bin}', '', 800, 650)
        c.SetRightMargin(0.14)
        shown.Draw('COLZ')

        peak = weighted_peak_2d(shown, peak_radius_bins)
        marker = None
        if draw_peak and peak:
            marker = root.TMarker(peak['dr'], peak['drphi'], 29)
            marker.SetMarkerSize(2.0)
            marker.Draw('SAME')

        label = root.TLatex()
        label.SetNDC(True)
        label.SetTextSize(0.035)
        label.DrawLatex(0.12, 0.94, f'Added through {pt_bin}, {charge}')

        inspection_objects.append((c, shown, marker, label))
        c.Draw()


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qplus_pt_0p4_0p7
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qminus_pt_0p4_0p7
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qplus_pt_0p7_1p2
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qminus_pt_0p7_1p2
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qplus_pt_1p2_1p8
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qminus_pt_1p2_1p8
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qplus_pt_1p8_5p0
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_qminus_pt_1p8_5p0


In [6]:
def make_map(name, title):
    h = root.TH2D(name, title, 3, -0.5, 2.5, 5, -0.5, 4.5)
    h.SetDirectory(0)
    h.GetXaxis().SetTitle('local #phi bin')
    h.GetYaxis().SetTitle('local layer bin')
    return h

maps = {}
for side in sides:
    for module in range(36):
        h_dr = make_map(f'h_deltaR_{side}_{module:02d}', f'{side}, module {module:02d}: #Delta r')
        h_dp = make_map(f'h_deltaRPhi_{side}_{module:02d}', f'{side}, module {module:02d}: r#Delta#phi')
        h_nc = make_map(f'h_nCategories_{side}_{module:02d}', f'{side}, module {module:02d}: categories')
        h_ok = make_map(f'h_valid_{side}_{module:02d}', f'{side}, module {module:02d}: valid')

        for phi_bin in range(3):
            for layer_bin in range(5):
                key = (side, module, phi_bin, layer_bin)
                bx, by = phi_bin+1, layer_bin+1
                h_nc.SetBinContent(bx, by, category_counts[key])
                point = combined_points[key]
                if point is None:
                    continue
                h_dr.SetBinContent(bx, by, point['dr'])
                h_dp.SetBinContent(bx, by, point['drphi'])
                h_ok.SetBinContent(bx, by, 1)

        maps[(side, module)] = {'dr':h_dr, 'drphi':h_dp, 'n_categories':h_nc, 'valid':h_ok}

print(f'Created {len(maps)} module maps in memory')


Created 72 module maps in memory


In [7]:
if write_output:
    out = root.TFile.Open(str(output_file), 'RECREATE')
    if not out or out.IsZombie():
        raise OSError(f'Cannot create {output_file}')

    for (side, module), module_maps in maps.items():
        side_dir = out.GetDirectory(side) or out.mkdir(side)
        module_dir = side_dir.mkdir(f'module_{module:02d}')
        module_dir.cd()
        for hist in module_maps.values():
            hist.Write()

    out.Close()
    print(f'Wrote {output_file}')
else:
    print('write_output = False: no ROOT map was written')


write_output = False: no ROOT map was written


In [8]:
plot_side = 'side0'
plot_module = 10

c_map = root.TCanvas('c_final_maps', '', 1200, 500)
c_map.Divide(2, 1)
c_map.cd(1)
maps[(plot_side, plot_module)]['dr'].Draw('COLZ TEXT')
c_map.cd(2)
maps[(plot_side, plot_module)]['drphi'].Draw('COLZ TEXT')
c_map.Draw()
